In [1]:
!git clone https://github.com/Priyansh-81/Architectural-Immunology.git
%cd Architectural-Immunology
!pip install -U transformers accelerate bitsandbytes sentencepiece datasets

Cloning into 'Architectural-Immunology'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 64 (delta 26), reused 42 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 34.00 KiB | 4.25 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/Architectural-Immunology
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully unin

In [2]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [3]:
import os
from getpass import getpass

token = getpass("HF token: ")

if token:
    os.environ["HF_TOKEN"] = token

HF token: ··········


In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [6]:
small_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

small_tokenizer = AutoTokenizer.from_pretrained(small_model_name)

small_model = AutoModelForCausalLM.from_pretrained(
    small_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Small model loaded")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Small model loaded


In [7]:
reasoning_model_name = "Qwen/Qwen2.5-3B-Instruct"

reasoning_tokenizer = AutoTokenizer.from_pretrained(reasoning_model_name)

reasoning_model = AutoModelForCausalLM.from_pretrained(
    reasoning_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Reasoning model loaded")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Reasoning model loaded


In [8]:
print(
    "GPU memory allocated:",
    torch.cuda.memory_allocated()/1e9,
    "GB"
)

GPU memory allocated: 7.159975936 GB


In [9]:
prompt = "Question: What is 2+2?\nAnswer:"

inputs = reasoning_tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = reasoning_model.generate(
    **inputs,
    max_new_tokens=30
)

print(
    reasoning_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Question: What is 2+2?
Answer: 2 + 2 is equal to 4.
A. True
B. False
Answer:
A

Question: If a car travels at


In [11]:
!pip install datasets pandas

In [14]:
from huggingface_hub import login
login()

In [15]:
from datasets import load_dataset

In [17]:
dataset = load_dataset("gaia-benchmark/GAIA", "2023_level1")

2023/test/metadata.level1.parquet:   0%|          | 0.00/27.3k [00:00<?, ?B/s]

2023/validation/metadata.level1.parquet:   0%|          | 0.00/39.5k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/93 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/53 [00:00<?, ? examples/s]

In [18]:
print(dataset)

DatasetDict({
    test: Dataset({
        features: ['task_id', 'Question', 'Level', 'Final answer', 'file_name', 'file_path', 'Annotator Metadata'],
        num_rows: 93
    })
    validation: Dataset({
        features: ['task_id', 'Question', 'Level', 'Final answer', 'file_name', 'file_path', 'Annotator Metadata'],
        num_rows: 53
    })
})
